![DB Academy](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/db-academy.png)


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Extra Material</strong>
  <div style="color:#333;">
    Bonus material, included for self-paced learning but outside the scope of live instruction due to time constraints.
  </div>
</div>



# Appendix A - Dashboard Caching Overview

Dataset optimization and caching - [AWS](https://docs.databricks.com/aws/en/dashboards/caching) | 
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/caching) | 
[GCP](https://docs.databricks.com/gcp/en/dashboards/caching)


## A. The Dashboard Cache and Query Cache
<br></br>

<div class="ch-wrap">

<style>
.ch-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }

/* Hero */
.ch-hero {
    width: 100%;
    max-width: 1100px;
    background: linear-gradient(135deg, #E3F2FD 0%, #BBDEFB 100%);
    border-radius: 10px;
    padding: 22px 28px;
    display: flex;
    align-items: center;
    gap: 18px;
    box-shadow: 0 2px 10px rgba(25,118,210,0.08);
    border-left: 8px solid #1976D2;
    box-sizing: border-box;
}
.ch-hero-icon {
    width: 54px;
    height: 54px;
    border-radius: 50%;
    background: #1976D2;
    color: #FFFFFF;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-shrink: 0;
}
.ch-hero-icon svg { width: 30px; height: 30px; display: block; }
.ch-hero-text { flex: 1; color: #0b2026; }
.ch-hero-title { font-size: 22pt; font-weight: 700; color: #0d47a1; line-height: 1.2; margin-bottom: 4px; }
.ch-hero-sub { font-size: 14pt; color: #0b2026; line-height: 1.45; }

/* Cache cards */
.ch-cache-row { display: flex; justify-content: center; align-items: stretch; gap: 18px; flex-wrap: wrap; max-width: 1100px; width: 100%; }
.ch-cache-card { flex: 1 1 460px; max-width: 540px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px 24px; position: relative; box-sizing: border-box; color: #0b2026; display: flex; flex-direction: column; }
.ch-cache-card::before { content: ""; position: absolute; top: 0; left: 0; width: 100%; height: 8px; border-top-left-radius: 8px; border-top-right-radius: 8px; }
.ch-cache-card.dashboard::before { background: #4299E0; }
.ch-cache-card.query::before { background: #00A972; }
.ch-cache-label { font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; margin-top: 6px; }
.ch-cache-card.dashboard .ch-cache-label { color: #4299E0; }
.ch-cache-card.query .ch-cache-label { color: #00A972; }
.ch-cache-title { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 2px 0 12px 0; line-height: 1.3; }
.ch-cache-card ul { margin: 0; padding-left: 20px; font-size: 14pt; color: #0b2026; line-height: 1.5; }
.ch-cache-card li { margin-bottom: 6px; }
.ch-cache-card li:last-child { margin-bottom: 0; }

/* Permissions sub-block */
.ch-perm-block { margin-top: 14px; padding: 12px 14px; background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; border-radius: 0 6px 6px 0; }
.ch-perm-eyebrow { font-weight: 700; font-size: 14pt; letter-spacing: 1.5px; text-transform: uppercase; color: #2272B4; margin-bottom: 6px; }
.ch-perm-block ul { margin: 0; padding-left: 18px; font-size: 14pt; line-height: 1.5; color: #0b2026; }
.ch-perm-block li { margin-bottom: 4px; }
.ch-perm-block li:last-child { margin-bottom: 0; }

/* Decision flow */
.ch-flow { width: 100%; max-width: 1100px; background: #FFFFFF; border-radius: 10px; padding: 18px 24px; box-sizing: border-box; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border: 1px solid #EEEDE9; }
.ch-flow-eyebrow { font-size: 14pt; font-weight: 700; color: #1B5162; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 4px; text-align: center; }
.ch-flow-sub { font-size: 14pt; color: #0b2026; text-align: center; margin-bottom: 14px; }
.ch-flow-svg svg { width: 100%; height: auto; max-width: 1000px; display: block; margin: 0 auto; }
</style>

<!-- Hero -->
<div class="ch-hero">
  <div class="ch-hero-icon">
    <svg viewBox="0 0 24 24" xmlns="http://www.w3.org/2000/svg" aria-hidden="true">
      <path d="M13 2L3 14h7l-1 8 10-12h-7l1-8z" fill="#FFFFFF"/>
    </svg>
  </div>
  <div class="ch-hero-text">
    <div class="ch-hero-title">Caching makes dashboards fast</div>
    <div class="ch-hero-sub">Two caches work together so the dashboard rarely has to actually run a query when someone opens it.</div>
  </div>
</div>

<!-- Two cache cards -->
<div class="ch-cache-row">

  <div class="ch-cache-card dashboard">
    <div class="ch-cache-label">Dashboard cache</div>
    <div class="ch-cache-title">Fast, best-effort, up to 24 hours</div>
    <ul>
      <li>Scoped to a single dashboard</li>
      <li>Keeps dataset results for up to 24 hours to speed up initial loads</li>
      <li>Can return data <strong>up to 24 hours stale</strong> (trades freshness for speed)</li>
    </ul>
    <div class="ch-perm-block">
      <div class="ch-perm-eyebrow">Permissions</div>
      <ul>
        <li><strong>Share data permissions:</strong> shared cache. All viewers see the same results. 
          <ul>
            <li><strong>Benefit:</strong> Warms the cache for everyone since its a shared cache. </li>
            <li><em>Generally recommended to use share data permissions when possible.</em> </li>
          </ul>
        <li><strong>Individual data permissions or draft:</strong> per-user cache. Each viewer's cache reflects their own data permissions.</li>
      </ul>
    </div>
  </div>

  <div class="ch-cache-card query">
    <div class="ch-cache-label">Query result cache</div>
    <div class="ch-cache-title">Never stale, warehouse-scoped</div>
    <ul>
      <li>Generic cache of SQL query results at the warehouse level</li>
      <li><strong>Warehouse-scoped</strong>, not dashboard-scoped</li>
      <li><strong>Never serves stale data:</strong> when underlying tables change, cached results are invalidated</li>
    </ul>
  </div>

</div>

<!-- Decision flow -->
<div class="ch-flow">
  <div class="ch-flow-eyebrow">How a dashboard decides what to use</div>
  <div class="ch-flow-sub">When a viewer opens a dashboard, it walks down this list until it gets a hit.</div>
  <div class="ch-flow-svg">

<svg viewBox="0 0 800 180" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="ch-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#5E7077"/>
    </marker>
  </defs>

  <!-- Step 1: Dashboard cache -->
  <rect x="20" y="40" width="220" height="100" rx="8" fill="#4299E0" stroke="#2c7bc7" stroke-width="2"/>
  <text x="130" y="64" text-anchor="middle" fill="#FFFFFF" font-size="11" font-weight="700" font-family="sans-serif" letter-spacing="2">STEP 1</text>
  <text x="130" y="92" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700" font-family="sans-serif">Dashboard cache</text>
  <text x="130" y="115" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Best-effort, 24h</text>
  <text x="130" y="131" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Scoped to this dashboard</text>

  <!-- Arrow 1 -> 2 -->
  <line x1="240" y1="90" x2="290" y2="90" stroke="#5E7077" stroke-width="2.5" marker-end="url(#ch-arr)"/>
  <text x="265" y="80" text-anchor="middle" fill="#5E7077" font-size="11" font-style="italic" font-family="sans-serif">if no hit</text>

  <!-- Step 2: Query result cache -->
  <rect x="290" y="40" width="220" height="100" rx="8" fill="#00A972" stroke="#02935E" stroke-width="2"/>
  <text x="400" y="64" text-anchor="middle" fill="#FFFFFF" font-size="11" font-weight="700" font-family="sans-serif" letter-spacing="2">STEP 2</text>
  <text x="400" y="92" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700" font-family="sans-serif">Query result cache</text>
  <text x="400" y="115" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Warehouse-scoped</text>
  <text x="400" y="131" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Never stale</text>

  <!-- Arrow 2 -> 3 -->
  <line x1="510" y1="90" x2="560" y2="90" stroke="#5E7077" stroke-width="2.5" marker-end="url(#ch-arr)"/>
  <text x="535" y="80" text-anchor="middle" fill="#5E7077" font-size="11" font-style="italic" font-family="sans-serif">if no hit</text>

  <!-- Step 3: Run query -->
  <rect x="560" y="40" width="220" height="100" rx="8" fill="#1B5162" stroke="#0F3A47" stroke-width="2"/>
  <text x="670" y="64" text-anchor="middle" fill="#FFFFFF" font-size="11" font-weight="700" font-family="sans-serif" letter-spacing="2">STEP 3</text>
  <text x="670" y="92" text-anchor="middle" fill="#FFFFFF" font-size="16" font-weight="700" font-family="sans-serif">Run the query</text>
  <text x="670" y="115" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Execute on the warehouse</text>
  <text x="670" y="131" text-anchor="middle" fill="#FFFFFF" font-size="12" font-family="sans-serif" opacity="0.95">Pay the full query cost</text>
</svg>

  </div>
</div>

</div>




### B. A Day in the Life of a Cached Dashboard (Example)
<br></br>

<div class="stp-wrap">

<style>
.stp-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }

/* Hero */
.stp-hero {
    width: 100%;
    max-width: 1100px;
    background: linear-gradient(135deg, #E3F2FD 0%, #BBDEFB 100%);
    border-radius: 10px;
    padding: 22px 28px;
    text-align: center;
    box-shadow: 0 2px 10px rgba(25,118,210,0.08);
    border-left: 8px solid #1976D2;
    box-sizing: border-box;
}
.stp-hero-title { font-size: 22pt; font-weight: 700; color: #0d47a1; line-height: 1.2; margin-bottom: 4px; }
.stp-hero-sub { font-size: 14pt; color: #0b2026; line-height: 1.45; }

/* Timeline SVG */
.stp-timeline {
    width: 100%;
    max-width: 1100px;
    background: #FFFFFF;
    border-radius: 10px;
    padding: 16px 24px;
    box-sizing: border-box;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    border: 1px solid #EEEDE9;
}
.stp-timeline svg { width: 100%; height: auto; max-width: 1000px; display: block; margin: 0 auto; }

/* Hint above the buttons */
.stp-hint {
    width: 100%;
    max-width: 1100px;
    text-align: center;
    font-size: 14pt;
    color: #5E7077;
    font-style: italic;
    margin-bottom: -4px;
}
.stp-hint strong { color: #0b2026; font-style: normal; font-weight: 700; }

/* Step button row */
.stp-row {
    display: block;
    white-space: nowrap;
    font-size: 0;
    width: 100%;
    max-width: 1100px;
    box-sizing: border-box;
}
.stp-box {
    display: inline-block;
    width: 23.5%;
    margin-right: 2%;
    min-height: 80px;
    background: #F9F7F4;
    border-top: 8px solid transparent;
    border-left: 2px solid transparent;
    border-right: 2px solid transparent;
    border-bottom: 2px solid transparent;
    border-radius: 8px;
    padding: 14px 12px;
    text-align: center;
    box-sizing: border-box;
    cursor: pointer;
    user-select: none;
    vertical-align: top;
    transition: transform 0.12s;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    white-space: normal;
}
.stp-box:last-child { margin-right: 0; }
.stp-box:hover { transform: translateY(-2px); }
.stp-box.active {
    background: #fff;
    border-left-color: var(--pc);
    border-right-color: var(--pc);
    border-bottom-color: var(--pc);
}
.stp-time {
    display: block;
    font-size: 16pt;
    font-weight: 700;
    color: #0b2026;
    line-height: 1.2;
    pointer-events: none;
}
.stp-dash {
    display: inline-block;
    margin-top: 6px;
    padding: 3px 14px;
    border-radius: 12px;
    font-size: 12pt;
    font-weight: 700;
    color: #FFFFFF;
    pointer-events: none;
    letter-spacing: 0.5px;
}
.stp-event {
    display: block;
    font-size: 13pt;
    color: #5E7077;
    line-height: 1.3;
    margin-top: 6px;
    pointer-events: none;
}

/* Detail wrap */
.stp-detail-wrap {
    overflow: hidden;
    max-height: 0;
    opacity: 0;
    transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
    margin-top: 0;
    width: 100%;
    max-width: 1100px;
}
.stp-detail-wrap.open {
    max-height: 700px;
    opacity: 1;
    margin-top: 12px;
}
.stp-detail-card {
    background: #F9F7F4;
    border-radius: 10px;
    padding: 22px 26px;
    box-sizing: border-box;
    border-top: 7px solid #ccc;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
}
.stp-detail-title {
    font-size: 17pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 10px;
    line-height: 1.3;
}
.stp-detail-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
    margin-bottom: 14px;
}
.stp-states { display: flex; gap: 10px; flex-wrap: wrap; }

/* Cache-type pills: solid background, white text */
.stp-pill {
    padding: 7px 16px;
    border-radius: 18px;
    font-size: 14pt;
    font-weight: 600;
    color: #FFFFFF;
    background: #5E7077;
}
.stp-pill-dash  { background: #1976D2; }
.stp-pill-query { background: #02935E; }
.stp-pill-muted { background: #5E7077; }

/* Why two caches */
.stp-why {
    display: flex;
    gap: 16px;
    max-width: 1100px;
    width: 100%;
    flex-wrap: wrap;
}
.stp-why-card {
    flex: 1 1 460px;
    background: #F9F7F4;
    border-radius: 8px;
    padding: 18px 22px;
    box-sizing: border-box;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    border-top: 6px solid #1976D2;
}
.stp-why-card.b { border-top-color: #02935E; }
.stp-why-card .swc-label {
    font-size: 14pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #1976D2;
    margin-bottom: 6px;
}
.stp-why-card.b .swc-label { color: #02935E; }
.stp-why-card .swc-text { font-size: 14pt; line-height: 1.5; color: #0b2026; }

/* Takeaway */
.stp-takeaway {
    width: 100%;
    max-width: 1100px;
    background: linear-gradient(135deg, #fff8e1 0%, #ffe0b2 100%);
    border-left: 8px solid #FFAB00;
    border-radius: 10px;
    padding: 22px 28px;
    box-sizing: border-box;
    box-shadow: 0 2px 10px rgba(245,158,11,0.10);
}
.stp-takeaway-eyebrow { font-size: 14pt; font-weight: 700; color: #e65100; letter-spacing: 1.5px; text-transform: uppercase; margin-bottom: 6px; }
.stp-takeaway-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; }
.stp-takeaway ul { margin: 0; padding-left: 20px; font-size: 14pt; color: #0b2026; line-height: 1.55; }
.stp-takeaway li { margin-bottom: 4px; }
</style>

<!-- Hero -->
<div class="stp-hero">
  <div class="stp-hero-title">A day in the life of a cached dashboard</div>
  <div class="stp-hero-sub">Click a moment in the day to see what happens to each cache.</div>
</div>

<!-- Timeline SVG (visual anchor) -->
<div class="stp-timeline">

<svg viewBox="0 0 1000 120" xmlns="http://www.w3.org/2000/svg">
  <line x1="80" y1="60" x2="920" y2="60" stroke="#5E7077" stroke-width="3"/>
  <line x1="80" y1="53" x2="80" y2="67" stroke="#5E7077" stroke-width="2"/>
  <line x1="920" y1="53" x2="920" y2="67" stroke="#5E7077" stroke-width="2"/>
  <text x="80" y="100" text-anchor="middle" fill="#5E7077" font-size="14" font-family="sans-serif">midnight</text>
  <text x="920" y="100" text-anchor="middle" fill="#5E7077" font-size="14" font-family="sans-serif">end of day</text>

  <circle cx="200" cy="60" r="14" fill="#FF5F46" stroke="#FFFFFF" stroke-width="3"/>
  <text x="200" y="35" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700" font-family="sans-serif">4:30 AM</text>

  <circle cx="430" cy="60" r="14" fill="#FF5F46" stroke="#FFFFFF" stroke-width="3"/>
  <text x="430" y="35" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700" font-family="sans-serif">9:00 AM</text>

  <circle cx="620" cy="60" r="14" fill="#1B3139" stroke="#FFFFFF" stroke-width="3"/>
  <text x="620" y="35" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700" font-family="sans-serif">10:30 AM</text>

  <circle cx="820" cy="60" r="14" fill="#FF5F46" stroke="#FFFFFF" stroke-width="3"/>
  <text x="820" y="35" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700" font-family="sans-serif">2:00 PM</text>
</svg>

</div>

<!-- Click hint -->
<div class="stp-hint"><strong>Click each time below</strong> to see what happens to each cache.</div>

<!-- Click-to-reveal step buttons -->
<div class="stp-row">
  <span class="stp-box" data-id="0" onclick="selectStep(0)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
    <span class="stp-time">4:30 AM</span>
    <span class="stp-dash" style="background:#FF5F46;">Dashboard A</span>
    <span class="stp-event">Scheduled refresh runs</span>
  </span><span class="stp-box" data-id="1" onclick="selectStep(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
    <span class="stp-time">9:00 AM</span>
    <span class="stp-dash" style="background:#FF5F46;">Dashboard A</span>
    <span class="stp-event">Kristi opens Dashboard A</span>
  </span><span class="stp-box" data-id="2" onclick="selectStep(2)" style="--pc:#1B3139; border-top-color:#1B3139;">
    <span class="stp-time">10:30 AM</span>
    <span class="stp-dash" style="background:#1B3139;">Dashboard B</span>
    <span class="stp-event">Panagiotis opens Dashboard B (same underlying table)</span>
  </span><span class="stp-box" data-id="3" onclick="selectStep(3)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
    <span class="stp-time">2:00 PM</span>
    <span class="stp-dash" style="background:#FF5F46;">Dashboard A</span>
    <span class="stp-event">Underlying table updates, no auto-refresh</span>
  </span>
</div>

<div class="stp-detail-wrap" id="stp-detail-wrap">
  <div class="stp-detail-card" id="stp-detail-card"></div>
</div>

<!-- Why two caches -->
<div class="stp-why">
  <div class="stp-why-card">
    <div class="swc-label">Why dashboard cache?</div>
    <div class="swc-text">Makes re-opening a dashboard <strong>instant</strong>, even across users. Accepts being a little stale to be fast.</div>
  </div>
  <div class="stp-why-card b">
    <div class="swc-label">Why query result cache?</div>
    <div class="swc-text">Enables <strong>cross-workload reuse</strong> without ever serving stale data. Two dashboards (or a dashboard plus an analyst's SQL) can share the same result.</div>
  </div>
</div>

<!-- Takeaway -->
<div class="stp-takeaway">
  <div class="stp-takeaway-eyebrow">What this means for you as a publisher</div>
  <div class="stp-takeaway-title">A scheduled refresh on a Share data permissions dashboard populates BOTH caches at once.</div>
  <ul>
    <li><strong>Dashboard cache:</strong> instant for this dashboard's viewers.</li>
    <li><strong>Query result cache:</strong> fast for any other dashboard or query that touches the same table.</li>
    <li>Without a schedule, the first viewer of the day pays the full query cost on a cold cache.</li>
  </ul>
</div>

</div>

<script>
var STEPS = [
  {
    title: "4:30 AM - Dashboard A scheduled refresh runs",
    color: "#FF5F46",
    text: "Your schedule runs every dataset query for <strong>Dashboard A</strong>. Both caches populate from cold.",
    states: [
      {cache: "dash",  text: "Dashboard cache: POPULATED"},
      {cache: "query", text: "Query result cache: POPULATED"}
    ]
  },
  {
    title: "9:00 AM - Kristi opens Dashboard A",
    color: "#FF5F46",
    text: "<ul style='margin: 0; padding-left: 22px;'><li style='margin-bottom: 10px;'>Step 1 of the lookup: <strong>dashboard cache HIT</strong>. Instant load.</li><li>The <strong>query result cache is not consulted</strong> because step 1 already returned a result.</li></ul>",
    states: [
      {cache: "dash",  text: "Dashboard cache: HIT"},
      {cache: "query", text: "Query result cache: still populated", muted: true}
    ]
  },
  {
    title: "10:30 AM - Panagiotis opens Dashboard B (different dashboard, same table)",
    color: "#1B3139",
    text: "<ul style='margin: 0; padding-left: 22px;'><li style='margin-bottom: 10px;'>Dashboard B's own <strong>dashboard cache is empty</strong> (step 1 misses, since this is a different dashboard).</li><li>However, the <strong>query result cache</strong> still has the result from Dashboard A's morning refresh. <strong>Step 2 hits</strong>. The widget loads fast even though Panagiotis has never opened Dashboard B before.</li></ul>",
    states: [
      {cache: "dash",  text: "Dashboard cache: EMPTY for Dashboard B", muted: true},
      {cache: "query", text: "Query result cache: HIT (cross-dashboard)"}
    ]
  },
  {
    title: "2:00 PM - Underlying table updates, but Dashboard A does NOT auto-refresh",
    color: "#FF5F46",
    text: "When the <strong>underlying table changes outside</strong> the schedule, two things happen at once.<ul style='margin: 10px 0; padding-left: 22px;'><li style='margin-bottom: 10px;'>Dashboard A's dashboard cache is <strong>not</strong> invalidated, it can keep serving the morning's result for the next ~21 hours (best-effort, 24h).</li><li>The query result cache is <strong>invalidated immediately</strong> (this also affects Dashboard B's cross-dashboard reuse).</li></ul><strong>Dashboard A viewers will see stale numbers</strong> until someone manually <strong>refreshes the dashboard</strong> or the next scheduled refresh fires.",
    states: [
      {cache: "dash",  text: "Dashboard A cache: STALE (still serving morning result)"},
      {cache: "query", text: "Query result cache: INVALIDATED", muted: true}
    ]
  }
];

var stpCurrent = null;

function selectStep(id) {
  var wrap = document.getElementById('stp-detail-wrap');
  var card = document.getElementById('stp-detail-card');
  var s = STEPS[id];

  document.querySelectorAll('.stp-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && stpCurrent !== id);
  });

  if (stpCurrent === id) {
    wrap.classList.remove('open');
    stpCurrent = null;
    return;
  }

  stpCurrent = id;

  var pillsHtml = s.states.map(function(st) {
    var cls = st.muted ? 'stp-pill-muted' : 'stp-pill-' + st.cache;
    return '<div class="stp-pill ' + cls + '">' + st.text + '</div>';
  }).join('');

  card.style.borderTopColor = s.color;
  card.innerHTML =
    '<div class="stp-detail-title" style="color:' + s.color + ';">' + s.title + '</div>'
    + '<div class="stp-detail-text">' + s.text + '</div>'
    + '<div class="stp-states">' + pillsHtml + '</div>';

  wrap.classList.add('open');
}
</script>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>